# 10 — Modelos de entrada y permanencia en la pobreza (bloque A)

Probabilidad de `pobre_th` entre los no pobres en `t` (entrada) y entre los pobres en `t` (permanencia). Discriminación (AUC-ROC, AUC-PR relativa) y calibración en grande (media vs. tasa, intercepto, pendiente, ECE) se reportan por separado; la selección usa log loss.

Universo 2003–2025, h=1 y h=4. Técnicas: línea base por decil, lineal regularizado y LightGBM. Cada una se entrena
con `imputacion_excluye` (referencia) e `imputacion_conserva`, y se evalúa siempre sobre el mismo conjunto limpio
(`par_valido_c`); en los folds con test histórico, además sobre los pares imputados. Los pares salen de
`validacion.cargar_pares_modelado` (solo train): **el holdout no se abre**. La lógica está en `src/modelos.py` y
`src/validacion.py`; las reglas, en `.claude/decisiones.md`.

`ETAPA = "grilla"` elige los hiperparámetros de GB y la penalización del lineal en la validación interna de los
6 últimos folds (nunca en sus tests). `ETAPA = "folds"` exige esa elección registrada en `modelos.HIPERPARAMETROS_GB`
y `modelos.PENALIZACION_LINEAL`, corre todos los folds, escribe `data/07_predicciones/` (cada fold ya escrito y
verificado se saltea, salvo `REHACER`) y las tablas de `data/meta/`.

In [ ]:
import json
import sys
import time
from pathlib import Path


def _raiz_repo(desde: Path | None = None) -> Path:
    actual = (desde or Path.cwd()).resolve()
    for candidato in (actual, *actual.parents):
        if (candidato / "pytest.ini").exists():
            return candidato
    raise FileNotFoundError(f"no se encontró la raíz del repo subiendo desde {actual}")


REPO = _raiz_repo()
sys.path.insert(0, str(REPO / "src"))

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

import constantes as cte
import modelos as mo
import pipeline_meta as pm
import universo_entrenamiento as ue
import validacion as va

ETAPA: str = "folds"
REHACER: bool = False
N_BOOT: int = 200
NOTEBOOK = "10_modelos_entrada_permanencia"
OBJETIVOS = [('entrada', 'simetrica'), ('permanencia', 'simetrica')]
HORIZONTES = [1, 4]

FEATURES = REPO / "data" / "06_features"
TRAYECTORIA = REPO / "data" / "05_trayectoria_distancia"
PREDICCIONES = REPO / "data" / "07_predicciones"
META = REPO / "data" / "meta"
PREFIJO = NOTEBOOK.split("_", 1)[1]
COMMIT_GIT = pm.obtener_commit_git(REPO)
COLOR_TECNICA = {"decil_ratio": "#2a78d6", "lineal_regularizado": "#eb6834", "gradient_boosting": "#1baf7a"}
pd.set_option("display.width", 250)
pd.set_option("display.max_rows", 400)
assert ETAPA in ("grilla", "folds")
t0_proceso = time.time()

## Pares de train y folds

Se cargan con `imputacion_conserva` (el superconjunto); cada criterio filtra su train con `validacion.mascara_train`.
Los folds son los del 08 para el universo 2003–2025.

In [ ]:
pares, folds = {}, {}
for h in HORIZONTES:
    p = va.cargar_pares_modelado(FEATURES, TRAYECTORIA, h, mo.columnas_extra_carga(h), criterio_imputacion="imputacion_conserva")
    pares[h] = p[ue.en_universo(p, "universo_2003_2025")].reset_index(drop=True)
    va.exigir_sin_holdout(pares[h])
    folds[h] = va.generar_folds(pares[h][va.mascara_train(pares[h])], h)
    print(f"h={h}: {len(pares[h]):,} pares, {len(mo.columnas_x(h))} columnas de X, folds {folds[h]['anio_test'].tolist()}")
FOLDS_SELECCION = {h: folds[h]["anio_test"].tolist()[-mo.N_FOLDS_SELECCION:] for h in HORIZONTES}
print("folds de selección:", FOLDS_SELECCION)

## Etapa grilla

LightGBM: las 8 combinaciones de `modelos.GRILLA_GB`, una elección por (objetivo, h); la variante log de la trayectoria
reusa la de la simétrica. Lineal: tipo de penalización × 7 fuerzas. Todo con `imputacion_excluye`, puntuado con la
pérdida de la validación interna (log loss / MSE).

In [ ]:
if ETAPA == "grilla":
    t0 = time.time()
    grilla_gb, grilla_lineal = [], []
    for h in HORIZONTES:
        for objetivo, variable in OBJETIVOS:
            if variable != "simetrica":
                continue
            grilla_gb.append(mo.elegir_hiperparametros_gb(pares[h], FOLDS_SELECCION[h], objetivo, h, variable))
            grilla_lineal.append(mo.elegir_penalizacion_lineal(pares[h], FOLDS_SELECCION[h], objetivo, h, variable))
            print(f"h={h} {objetivo}: {time.time() - t0:.0f}s", flush=True)
    grilla_gb, grilla_lineal = pd.concat(grilla_gb, ignore_index=True), pd.concat(grilla_lineal, ignore_index=True)
    grilla_gb.to_csv(META / f"{PREFIJO}_grilla_gb.csv", index=False)
    grilla_lineal.to_csv(META / f"{PREFIJO}_grilla_lineal.csv", index=False)
    resumen_gb, resumen_lineal = mo.resumir_grilla_gb(grilla_gb), mo.resumir_penalizacion_lineal(grilla_lineal)
    display(resumen_gb.round(5))
    display(resumen_lineal.round(5))
    print("elegidas GB:")
    display(resumen_gb[resumen_gb["elegida"]])
    pm.registrar_corrida(META / "corridas.csv", NOTEBOOK,
                         {"ETAPA": ETAPA, "folds_seleccion": FOLDS_SELECCION, "grilla_gb": mo.GRILLA_GB,
                          "fuerzas_lineal": mo.FUERZAS_LINEAL},
                         sum(len(v) for v in FOLDS_SELECCION.values()), len(grilla_gb) + len(grilla_lineal),
                         time.time() - t0_proceso, COMMIT_GIT)

## Etapa folds: entrenamiento y predicciones

Orden por (objetivo, h): decil, lineal (excluye elige la fuerza por fold; conserva la reusa), GB (los dos criterios con
los mismos hiperparámetros, cada uno con su early stopping). Folds sin validación interna utilizable quedan omitidos.

In [ ]:
if ETAPA == "folds":
    t0 = time.time()
    ajustes, configuraciones = [], {}
    for h in HORIZONTES:
        anios = folds[h]["anio_test"].tolist()
        for objetivo, variable in OBJETIVOS:
            lista = mo.configuraciones_bloque_a(objetivo, variable, h)
            configuraciones[(objetivo, variable, h)] = lista
            fuerzas = None
            for configuracion in lista:
                registro = mo.correr_configuracion(pares[h], anios, configuracion, PREDICCIONES, REPO, NOTEBOOK, REHACER,
                                                   fuerzas if configuracion.criterio_imputacion == "imputacion_conserva" else None)
                if configuracion.tecnica == "lineal_regularizado" and configuracion.criterio_imputacion == "imputacion_excluye":
                    fuerzas = (registro.dropna(subset=["fuerza"]).set_index("anio_test")["fuerza"].to_dict()
                               if "fuerza" in registro else {})
                ajustes.append(registro.assign(objetivo_modelado=objetivo, variable=variable, h=h,
                                               tecnica=configuracion.tecnica, criterio_imputacion=configuracion.criterio_imputacion))
                print(f"h={h} {objetivo}/{variable} {configuracion.nombre}: {time.time() - t0:.0f}s", flush=True)
    ajustes = pd.concat(ajustes, ignore_index=True)
    for columna in ("error", "omitido"):
        if columna not in ajustes:
            ajustes[columna] = pd.NA
    errores = ajustes[ajustes["error"].notna()]
    print(f"folds con error: {len(errores)}")
    display(errores[["objetivo_modelado", "variable", "h", "configuracion", "anio_test", "error"]])
    ajustes["train_chico"] = ajustes["n_final"] < mo.UMBRAL_TRAIN_CHICO
    ajustes.to_csv(META / f"{PREFIJO}_ajuste_fold.csv", index=False)
    columnas = ["n_es", "n_validacion", "n_final", "mejor_iteracion", "n_iteraciones_final", "fuerza"]
    display(ajustes.pivot_table(index=["objetivo_modelado", "variable", "h", "anio_test"],
                                columns=["tecnica", "criterio_imputacion"], values=["n_final"], aggfunc="first"))
    display(ajustes[ajustes["tecnica"] != "decil_ratio"].set_index(
        ["objetivo_modelado", "variable", "h", "tecnica", "criterio_imputacion", "anio_test"])[
        [c for c in columnas if c in ajustes] + ["reanudado"]].round(1))

### Verificación de las predicciones fuera de muestra

In [ ]:
if ETAPA == "folds":
    escritas = 0
    for lista in configuraciones.values():
        for configuracion in lista:
            escritas += len(mo.leer_predicciones(PREDICCIONES, configuracion))
    print(f"verificador OK: {escritas:,} predicciones de {sum(len(l) for l in configuraciones.values())} configuraciones")

## Métricas por fold contra el decil y diferencias conserva − excluye

Un bootstrap por vivienda por (fold, objetivo, conjunto) con `N_BOOT` réplicas, compartido por todas las
configuraciones: modelo − decil y conserva − excluye salen pareados. La lectura ponderada
(`peso_normalizado_trimestre`) va sin IC.

In [ ]:
if ETAPA == "folds":
    t0 = time.time()
    evaluaciones, errores_evaluacion = [], []
    for (objetivo, variable, h), lista in configuraciones.items():
        historicos = folds[h].loc[folds[h]["era_test"] == "historica", "anio_test"].tolist()
        for anio in folds[h]["anio_test"]:
            lecturas = [("limpio", "sin_ponderar", N_BOOT), ("limpio", "ponderado_trimestre", 0)]
            if anio in historicos:
                lecturas.append(("imputados", "sin_ponderar", N_BOOT))
            for conjunto, ponderacion, n_boot in lecturas:
                try:
                    evaluaciones.append(mo.evaluar_configuraciones_fold(pares[h], PREDICCIONES, lista, int(anio), conjunto,
                                                                        ponderacion, n_boot))
                except (va.FugaHoldoutError, ValueError):
                    raise
                except Exception as error:
                    errores_evaluacion.append({"objetivo_modelado": objetivo, "variable": variable, "h": h, "anio_test": anio,
                                               "conjunto_evaluacion": conjunto, "ponderacion": ponderacion,
                                               "error": f"{type(error).__name__}: {error}"})
        print(f"h={h} {objetivo}/{variable}: {time.time() - t0:.0f}s", flush=True)
    metricas = pd.concat([e for e in evaluaciones if not e.empty], ignore_index=True)
    marcas = pd.concat([f.assign(h=h) for h, f in folds.items()])[["h", "anio_test", "contexto_fold", "destino_en_contexto",
                                                                   "periodo_intervenido"]]
    metricas = metricas.merge(marcas, on=["h", "anio_test"], how="left")
    metricas["salto_de_era"] = metricas["contexto_fold"] == "salto_de_era"
    metricas["contexto_marcado"] = metricas["contexto_fold"].fillna("").ne("") & ~metricas["salto_de_era"]
    chico = ajustes.groupby(["objetivo_modelado", "variable", "h", "anio_test"])["train_chico"].any().rename("train_chico")
    metricas = metricas.merge(chico.reset_index(), on=["objetivo_modelado", "variable", "h", "anio_test"], how="left")
    metricas["train_chico"] = metricas["train_chico"].fillna(False).astype(bool)
    metricas.to_csv(META / f"{PREFIJO}_metricas_fold.csv", index=False)
    diferencias = metricas[metricas["configuracion"].str.contains("conserva_menos_excluye")]
    diferencias = diferencias.merge(
        ajustes[ajustes["criterio_imputacion"] == "imputacion_conserva"].groupby(["objetivo_modelado", "variable", "h", "anio_test"])
        ["fraccion_historica_train"].max().reset_index(), on=["objetivo_modelado", "variable", "h", "anio_test"], how="left")
    diferencias.to_csv(META / f"{PREFIJO}_diferencias_criterio.csv", index=False)
    print(f"{len(metricas):,} filas de métricas; evaluaciones con error: {len(errores_evaluacion)}")
    display(pd.DataFrame(errores_evaluacion))

### Skill contra el decil por fold

In [ ]:
METRICA_SKILL = {"trayectoria": "skill_vs_base", "entrada": "brier_skill_vs_base", "permanencia": "brier_skill_vs_base"}
if ETAPA == "folds":
    principal = metricas[(metricas["conjunto_evaluacion"] == "limpio") & (metricas["ponderacion"] == "sin_ponderar")]
    for (objetivo, variable), _ in principal.groupby(["objetivo_modelado", "variable"]):
        sub = principal[(principal["objetivo_modelado"] == objetivo) & (principal["variable"] == variable)
                        & (principal["metrica"] == METRICA_SKILL[objetivo]) & ~principal["configuracion"].str.startswith("decil")]
        print(objetivo, variable)
        display(sub.pivot_table(index=["h", "anio_test", "contexto_fold", "train_chico"], columns="configuracion",
                                values=["valor"], aggfunc="first").round(3))

### Diferencia conserva − excluye a lo largo de los folds

Contra la fracción histórica del train de cada fold (no contra el año): se espera que se achique a medida que el train se
vuelve mayormente regular.

In [ ]:
METRICA_DIFERENCIA = {"trayectoria": "rmse", "entrada": "log_loss", "permanencia": "log_loss"}
if ETAPA == "folds":
    sub = diferencias[(diferencias["conjunto_evaluacion"] == "limpio") & (diferencias["ponderacion"] == "sin_ponderar")
                      & (diferencias["metrica"] == diferencias["objetivo_modelado"].map(METRICA_DIFERENCIA))]
    grupos = list(sub.groupby(["objetivo_modelado", "variable", "h"]))
    figura, ejes = plt.subplots(1, len(grupos), figsize=(4.2 * len(grupos), 3.6), squeeze=False, sharey=False)
    for eje, ((objetivo, variable, h), g) in zip(ejes[0], grupos):
        for configuracion, gg in g.groupby("configuracion"):
            tecnica = configuracion.split("|")[0]
            gg = gg.sort_values("fraccion_historica_train")
            eje.errorbar(gg["fraccion_historica_train"], gg["valor"], yerr=[gg["valor"] - gg["ic95_low"], gg["ic95_high"] - gg["valor"]],
                         color=COLOR_TECNICA[tecnica], marker="o", markersize=4, linewidth=2, capsize=0, label=tecnica)
        eje.axhline(0, color="#888780", linewidth=1)
        eje.set_title(f"{objetivo} {variable} h={h}", fontsize=10)
        eje.set_xlabel("fracción histórica del train")
        eje.set_ylabel(f"conserva − excluye ({METRICA_DIFERENCIA[objetivo]})")
        eje.grid(alpha=0.2)
    ejes[0][0].legend(fontsize=8, frameon=False)
    figura.tight_layout()
    plt.show()
    pendiente = sub.groupby(["objetivo_modelado", "variable", "h", "configuracion"]).apply(
        lambda g: pd.Series({"pendiente_vs_fraccion_historica": np.polyfit(g["fraccion_historica_train"], g["valor"], 1)[0]
                             if g["fraccion_historica_train"].nunique() > 1 else np.nan,
                             "diferencia_media_folds_historicos": g.loc[g["fraccion_historica_train"] == 1, "valor"].mean(),
                             "diferencia_media_folds_regulares": g.loc[g["fraccion_historica_train"] < 1, "valor"].mean(),
                             "folds_ic_excluye_0": int(((g["ic95_low"] > 0) | (g["ic95_high"] < 0)).sum()),
                             "n_folds": len(g)}), include_groups=False)
    display(pendiente.round(4))

### Estabilidad entre folds (objetivo 2)

Dispersión del skill entre folds, excluidos los de train chico (< 50.000 pares), y media en folds con y sin cada marca.

In [ ]:
if ETAPA == "folds":
    skill = principal[principal["metrica"] == principal["objetivo_modelado"].map(METRICA_SKILL)]
    estabilidad = mo.resumen_estabilidad(skill)
    estabilidad.to_csv(META / f"{PREFIJO}_estabilidad.csv", index=False)
    display(estabilidad.set_index(["objetivo_modelado", "variable", "h", "configuracion"]).drop(columns="metrica").round(3))
    print("folds con train chico:", sorted(set(map(tuple, ajustes.loc[ajustes["train_chico"], ["h", "anio_test"]].to_numpy()))))

### Calibración de permanencia: pendiente por fold contra la del decil

Seguimiento de la anomalía del 08 (pendiente < 1 del decil en el universo 2003–2025): pendiente de calibración de GB, lineal y
decil por fold, en los dos criterios. Pendiente 1 = bien calibrado; < 1 = predicciones demasiado extremas.

In [ ]:
if ETAPA == "folds":
    calibracion = principal[(principal["objetivo_modelado"] == "permanencia")
                            & principal["metrica"].isin(["pendiente_calibracion", "intercepto_calibracion", "ece",
                                                         "auc_roc", "auc_pr_relativa"])
                            & ~principal["configuracion"].str.contains("menos")]
    calibracion.to_csv(META / "modelos_permanencia_calibracion.csv", index=False)
    pendientes = calibracion[calibracion["metrica"] == "pendiente_calibracion"]
    display(pendientes.pivot_table(index=["h", "anio_test", "contexto_fold"], columns="configuracion", values="valor",
                                   aggfunc="first").round(3))
    figura, ejes = plt.subplots(1, 2, figsize=(10, 3.6), squeeze=False)
    for eje, (h, g) in zip(ejes[0], pendientes.groupby("h")):
        for configuracion, gg in g.groupby("configuracion"):
            tecnica, criterio = configuracion.split("|")
            gg = gg.sort_values("anio_test")
            eje.plot(gg["anio_test"], gg["valor"], color=COLOR_TECNICA[tecnica], linewidth=2, marker="o", markersize=4,
                     linestyle="-" if criterio == "imputacion_excluye" else "--", label=f"{tecnica} ({criterio.split('_')[1]})")
        eje.axhline(1, color="#888780", linewidth=1)
        eje.set_title(f"permanencia h={h}: pendiente de calibración", fontsize=10)
        eje.set_xlabel("año de test")
        eje.grid(alpha=0.2)
    ejes[0][0].legend(fontsize=7, frameon=False)
    figura.tight_layout()
    plt.show()

## Candado y registro de la corrida

In [ ]:
corridas = pd.read_csv(META / "corridas.csv", dtype=str)
aperturas = sum(json.loads(p).get("evento") == va.EVENTO_APERTURA for p in corridas["parametros_json"].dropna())
print(f"eventos {va.EVENTO_APERTURA} en corridas.csv: {aperturas}")
assert aperturas == 0
if ETAPA == "folds":
    pm.registrar_corrida(META / "corridas.csv", NOTEBOOK,
                         {"ETAPA": ETAPA, "REHACER": REHACER, "N_BOOT": N_BOOT,
                          "hiperparametros_gb": {f"{k[0]}|{k[1]}": v for k, v in mo.HIPERPARAMETROS_GB.items()},
                          "penalizacion_lineal": {f"{k[0]}|{k[1]}": v for k, v in mo.PENALIZACION_LINEAL.items()},
                          "version_x": mo.VERSION_X, "hilos": mo.HILOS, "folds_con_error": len(errores),
                          "evaluaciones_con_error": len(errores_evaluacion)},
                         int(sum(len(f) for f in folds.values())), escritas, time.time() - t0_proceso, COMMIT_GIT)